In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

In [ ]:
tokenizer.encode("I love Pizza")

In [ ]:
tokenizer.decode([306, 5360, 349, 24990])

In [ ]:
from transformers import AutoTokenizer, BertModel
import torch

def generate_word_embeddings_bert(text, model_name="bert-base-uncased"):
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = BertModel.from_pretrained(model_name)

        # Tokenize the input text
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

        # Get the word embeddings
        with torch.no_grad():
            outputs = model(**inputs)

        # Extract embeddings and corresponding words
        embeddings = outputs.last_hidden_state
        input_ids = inputs["input_ids"][0]
        words = [tokenizer.decode(token_id) for token_id in input_ids]

        # Create a dictionary of word embeddings
        word_embeddings = {}
        for i, word in enumerate(words):
            if word not in ['[CLS]', '[SEP]', '[PAD]']:
                word_embeddings[word] = embeddings[0, i, :].numpy()

        return word_embeddings

    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [ ]:
# Example usage:
text1 = "I was walking on the bank of river"
text2 = "I went to the bank to withdraw money"
embeddings = generate_word_embeddings_bert(text2)

if embeddings:
    for word, embedding in embeddings.items():
        print(f"{word}: {embedding[:5]}...")

**Note:** Example outputs for the word *bank* (from two different contexts):

```
bank: [-0.16297182 -0.1512853  -0.08349326 -0.42740345 -0.15986079]...
bank: [ 0.43778786 -0.44205457  0.01638133 -0.08081906  0.83776623]...
```


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [ ]:
# Training data
sentences = [
    "the fluffy cat that was sleeping under the table woke up and stretched",
    "the old dog that had been chasing squirrels all day finally rested",
    "the small bird that flew away from the nest returned to its mother",
    "the big house that stood on the hill overlooked the entire valley",
    "the curious child who found a hidden door opened it carefully",
    "the talented musician who played the violin performed a beautiful piece",
    "the adventurous traveler who explored the ancient ruins discovered a secret passage",
    "the dedicated scientist who studied the rare flower found a new species"
]

In [ ]:
# Create vocabulary
words = set()
for sentence in sentences:
    for word in sentence.split():
        words.add(word)
words = list(words)

In [ ]:
words

In [ ]:
word_to_index = {word: i for i, word in enumerate(words)}
index_to_word = {i: word for i, word in enumerate(words)}

In [ ]:
word_to_index

In [ ]:
index_to_word

In [ ]:
vocab_size = len(words)

In [ ]:
vocab_size

In [ ]:
X_pairs = []
y_next = []

In [ ]:
for sentence in sentences:
    sentence_words = sentence.split()
    for i in range(len(sentence_words) - 2):
        X_pairs.append([word_to_index[sentence_words[i]], word_to_index[sentence_words[i + 1]]])
        y_next.append(word_to_index[sentence_words[i + 2]])

In [ ]:
X_pairs

In [ ]:
y_next

In [ ]:
X_pairs = np.array(X_pairs)
y_next = np.array(y_next)

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X_pairs, y_next, test_size=0.2, random_state=42)

In [ ]:
# Model
embedding_dim = 5

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])


In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(X_train, y_train, epochs=20, verbose=1)

In [ ]:
model.summary()

In [ ]:
input_pair = np.array([[word_to_index["the"], word_to_index["cat"]]])  # (1, 2)
predictions = model.predict(input_pair)
predictions

In [ ]:
predicted_index = np.argmax(predictions[0])
print(f"RNN: Predicted next word for 'the cat': {index_to_word[predicted_index]}")